# Assurance Gaps in an Integrated Safety and Cybersecurity Case
## For an AI-Based Perception Component in Highly Automated Driving

**Authors:** Milin Patel, Rolf Jung  
**Affiliation:** Kempten University of Applied Sciences  
**Venue:** SafeComp 2026 WAISE Workshop

---

This notebook reproduces the complete five-step constructive integration methodology
and all publication results. It can be run in Google Colab or any Jupyter environment.

**What this notebook does:**
1. Installs the repository and dependencies
2. Executes each methodology step with detailed explanation
3. Generates all tables, figures, and analysis outputs
4. Displays publication-ready results inline

**Methodology overview:**
- **Step 1:** Claim extraction from 5 automotive standards (52 clauses → 40 claims)
- **Step 2:** Lifecycle-phase mapping across 6 development phases
- **Step 3:** Integrated GSN construction extending ISO/PAS 8800 Annex B (6 → 9 goals)
- **Step 4:** Junction-point analysis identifying requirement inconsistencies
- **Step 5:** Assurance gap identification and classification
- **Evaluation:** Synthetic CARLA evaluation under 25 weather conditions

## 0. Environment Setup

In [ ]:
# Install the repository (run this cell once)
import os
import sys

# Clone the repository if running in Colab
if 'google.colab' in sys.modules:
    if not os.path.exists('Assurance-Gaps-in-an-Integrated-Safety-and-Cybersecurity-Case'):
        !git clone https://github.com/milinpatel07/Assurance-Gaps-in-an-Integrated-Safety-and-Cybersecurity-Case.git
    os.chdir('Assurance-Gaps-in-an-Integrated-Safety-and-Cybersecurity-Case')
    !pip install -e ".[dev]" -q
    !apt-get install -y graphviz -qq
else:
    # Running locally — ensure we're in the repo root
    if os.path.basename(os.getcwd()) == 'notebooks':
        os.chdir('..')
    # Install if needed
    !pip install -e ".[dev]" -q

print(f"Working directory: {os.getcwd()}")
print("Setup complete.")

In [ ]:
# Core imports
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, HTML, Markdown

# Project imports
from src.standards.registry import StandardsRegistry
from src.gsn.integrated_pattern import build_integrated_gsn
from src.analysis.inconsistencies import InconsistencyCatalogue
from src.analysis.gaps import GapClassification
from src.analysis.evidence_convergence import EvidenceConvergenceAnalysis
from src.evaluation.carla_evaluator import generate_synthetic_evaluation
from src.evaluation.weather_conditions import generate_weather_grid, compute_triggering_coverage

# Set deterministic seed
SEED = 42
SCENES_PER_WEATHER = 50
np.random.seed(SEED)

print(f"Random seed: {SEED}")
print(f"Scenes per weather condition: {SCENES_PER_WEATHER}")

---
## Step 1: Claim Extraction from Standard Clauses

We analyse five automotive standards applicable to an AI-based LiDAR perception component:

| Standard | Year | Scope |
|----------|------|-------|
| ISO 26262 | 2018 | Functional safety |
| ISO 21448 | 2022 | Safety of the intended functionality (SOTIF) |
| ISO/SAE 21434 | 2021 | Cybersecurity engineering |
| ISO/PAS 8800 | 2024 | Safety and artificial intelligence |
| ISO/IEC TR 5469 | 2024 | Functional safety and AI systems |

From each standard, we extract **claims** — normative statements that contribute
to the safety/security argument. Each claim is traced to a specific clause and
mapped to a GSN goal node.

In [ ]:
# Step 1: Build the standards registry and extract claims
registry = StandardsRegistry()

print("Standards Framework")
print("=" * 70)
total_claims = 0
total_clauses = 0

for std in registry.all_standards:
    n_claims = len(std.claims)
    n_clauses = len(std.clauses)
    total_claims += n_claims
    total_clauses += n_clauses
    print(f"\n  {std.standard_id} ({std.year}): {std.full_name}")
    print(f"    Clauses analysed: {n_clauses}")
    print(f"    Claims extracted: {n_claims}")
    for c in std.claims[:3]:  # Show first 3 claims per standard
        print(f"      [{c.claim_id}] → {c.gsn_goal}: {c.text[:65]}...")
    if n_claims > 3:
        print(f"      ... and {n_claims - 3} more claims")

print(f"\n{'=' * 70}")
print(f"Total: {total_clauses} clauses → {total_claims} claims from {len(registry.all_standards)} standards")

---
## Step 2: Lifecycle-Phase Mapping

Claims are mapped to six lifecycle phases (combining traditional safety lifecycle
with AI-specific phases):

1. **Concept / Requirements** — HARA, TARA, requirement specification
2. **Design / Training** — Architecture design, model training
3. **Verification & Validation** — Testing, V&V activities
4. **Integration / Deployment** — System integration, release
5. **Operation / Monitoring** — Runtime monitoring, incident response
6. **Modification / Re-assurance** — OTA updates, re-certification

The coverage matrix reveals which standards address which phases, and where
coverage gaps exist.

In [ ]:
# Step 2: Compute the coverage matrix
coverage_matrix = registry.compute_coverage_matrix()

# Display as a formatted table
print("Coverage Matrix: Number of Applicable Clauses per Phase")
print("=" * 90)

phases = list(list(coverage_matrix.values())[0].keys())
header = f"{'Standard':<15}" + "".join(f"{p[:20]:<22}" for p in phases)
print(header)
print("-" * 90)

for std_id, phase_counts in coverage_matrix.items():
    row = f"{std_id:<15}"
    for phase in phases:
        count = phase_counts[phase]
        row += f"{count if count > 0 else '---':<22}"
    print(row)

print(f"\nKey observation: ISO 21448 has the highest coverage in V&V (6 clauses)")
print(f"and Concept/Requirements (6 clauses), while ISO 26262 has no")
print(f"Operation/Monitoring clauses for the AI perception component.")

In [ ]:
# Visualize the coverage heatmap
from src.visualization.coverage_plots import plot_coverage_heatmap

fig, ax = plt.subplots(figsize=(14, 5))

standards = list(coverage_matrix.keys())
phases = list(list(coverage_matrix.values())[0].keys())
data = np.array([[coverage_matrix[s][p] for p in phases] for s in standards])

im = ax.imshow(data, cmap='YlOrRd', aspect='auto')
ax.set_xticks(range(len(phases)))
ax.set_xticklabels(phases, rotation=30, ha='right')
ax.set_yticks(range(len(standards)))
ax.set_yticklabels(standards)

for i in range(len(standards)):
    for j in range(len(phases)):
        val = data[i, j]
        text = str(int(val)) if val > 0 else '---'
        color = 'white' if val > 3 else 'black'
        ax.text(j, i, text, ha='center', va='center', color=color, fontweight='bold')

plt.colorbar(im, label='Number of applicable clauses')
ax.set_title('Clause Applicability per Lifecycle Phase (Table 2)', fontweight='bold')
plt.tight_layout()
plt.show()

---
## Step 3: Integrated GSN Construction

We construct an integrated Goal Structuring Notation (GSN) argument pattern
by extending the base pattern from ISO/PAS 8800 Annex B.

The base pattern has 6 goals. We extend it with:
- **G7:** SOTIF sufficiency (from ISO 21448)
- **G8:** Cybersecurity sufficiency (from ISO/SAE 21434)
- **G9:** Undeveloped — no standard addresses combined re-assurance

**Junction points** are goals where claims from 2+ standards converge.
These are where inconsistencies arise.

In [ ]:
# Step 3: Build the integrated GSN
gsn = build_integrated_gsn()
stats = gsn.compute_statistics()

print("Integrated GSN Statistics")
print("=" * 50)
for key, value in stats.items():
    print(f"  {key}: {value}")

print(f"\nJunction Points (goals with claims from 2+ standards):")
print("-" * 60)
for jp in gsn.get_junction_points():
    stds = ', '.join(jp.source_standards)
    print(f"  {jp.element_id}: [{stds}] ({len(jp.source_standards)} standards)")

print(f"\nUndeveloped Goals (identified gaps):")
print("-" * 60)
for ug in gsn.get_undeveloped_goals():
    print(f"  {ug.element_id}: {ug.text}")

In [ ]:
# Visualize the goal density (which standards contribute to each goal)
density = registry.compute_goal_density()

print("Standard Coverage Density per GSN Goal Node (Table 4)")
print("=" * 75)

all_stds = list(list(density.values())[0].keys())
header = f"{'Goal':<6}" + "".join(f"{s:<14}" for s in all_stds) + "Active"
print(header)
print("-" * 75)

for goal, stds in density.items():
    active = sum(1 for v in stds.values() if v)
    row = f"{goal:<6}"
    for s in all_stds:
        row += f"{'Y':<14}" if stds[s] else f"{'---':<14}"
    row += str(active)
    print(row)

print(f"\nG5 (V&V sufficiency) has the highest density: all 4 normative standards.")
print(f"This is where the evidence type asymmetry (I-2) manifests.")

---
## Step 4: Junction-Point Analysis (Inconsistencies)

At each junction point, claims from different standards may conflict.
We classify inconsistencies into three types:

- **Structural (S):** Standards impose incompatible framework structures
- **Terminological (T):** Same terms defined differently across standards
- **Methodological (M):** Standards prescribe conflicting methods for the same activity

In [ ]:
# Step 4: Inconsistency analysis
catalogue = InconsistencyCatalogue()

print("Requirement Inconsistencies at Junction Points")
print("=" * 90)

stats = catalogue.summary_statistics()
print(f"Total: {stats['total']} ({stats['structural']} structural, "
      f"{stats['terminological']} terminological, {stats['methodological']} methodological)")
print()

for inc in catalogue.inconsistencies:
    type_label = {'structural': 'S', 'terminological': 'T', 'methodological': 'M'}
    t = type_label[inc.inconsistency_type.value]
    nodes = ', '.join(inc.gsn_nodes)
    stds = ', '.join(inc.standards_involved)
    print(f"  {inc.inconsistency_id} [{t}] at {nodes}")
    print(f"    Standards: {stds}")
    print(f"    {inc.description}")
    print()

In [ ]:
# Visualize inconsistency distribution across GSN nodes
from collections import defaultdict

node_types = defaultdict(lambda: defaultdict(int))
for inc in catalogue.inconsistencies:
    for node in inc.gsn_nodes:
        node_types[node][inc.inconsistency_type.value] += 1

nodes = sorted(node_types.keys())
types = ['structural', 'terminological', 'methodological']
colors = ['#E74C3C', '#F39C12', '#3498DB']

fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(nodes))
width = 0.25

for i, (t, c) in enumerate(zip(types, colors)):
    values = [node_types[n][t] for n in nodes]
    ax.bar(x + i * width, values, width, label=t.capitalize(), color=c)

ax.set_xlabel('GSN Goal Node')
ax.set_ylabel('Number of Inconsistencies')
ax.set_title('Distribution of Requirement Inconsistencies across GSN Nodes', fontweight='bold')
ax.set_xticks(x + width)
ax.set_xticklabels(nodes)
ax.legend()
plt.tight_layout()
plt.show()

---
## Step 5: Assurance Gap Identification

Assurance gaps are places in the integrated argument where the evidence
is insufficient. We classify gaps into three types:

- **Missing claim:** No standard makes a claim for a required property
- **Missing evidence:** A claim exists but no evidence procedure is defined
- **Unresolved inconsistency:** Conflicting claims that cannot be resolved

Critically, some gaps are **integration-induced** — they only appear when
standards are combined, and would not be visible from any single standard alone.

In [ ]:
# Step 5: Gap classification
gaps = GapClassification()

print("Assurance Gaps in the Integrated Argument")
print("=" * 90)

gap_stats = gaps.summary_statistics()
print(f"Total: {gap_stats['total']} gaps")
print(f"  Missing claim:            {gap_stats['missing_claim']}")
print(f"  Missing evidence:         {gap_stats['missing_evidence']}")
print(f"  Unresolved inconsistency: {gap_stats['unresolved_inconsistency']}")
print(f"  Integration-induced:      {gap_stats['integration_induced']}")
print()

for g in gaps.gaps:
    marker = ' [INTEGRATION-INDUCED]' if g.integration_induced else ''
    print(f"  {g.gap_id}: {g.description}{marker}")
    print(f"    Type: {g.gap_type.value} | Phase: {g.lifecycle_phase.display_name}")
    print()

In [ ]:
# Visualize gap distribution across lifecycle phases
phase_gaps = defaultdict(lambda: {'standard': 0, 'integration': 0})
for g in gaps.gaps:
    key = g.lifecycle_phase.display_name
    if g.integration_induced:
        phase_gaps[key]['integration'] += 1
    else:
        phase_gaps[key]['standard'] += 1

phases_with_gaps = sorted(phase_gaps.keys())
fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(phases_with_gaps))

std_vals = [phase_gaps[p]['standard'] for p in phases_with_gaps]
int_vals = [phase_gaps[p]['integration'] for p in phases_with_gaps]

ax.bar(x - 0.15, std_vals, 0.3, label='Standard gap', color='#7B68EE')
ax.bar(x + 0.15, int_vals, 0.3, label='Integration-induced', color='#FF6347', hatch='//')

ax.set_xlabel('Lifecycle Phase')
ax.set_ylabel('Number of Gaps')
ax.set_title('Assurance Gap Distribution across Lifecycle Phases', fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(phases_with_gaps, rotation=15, ha='right')
ax.legend()
plt.tight_layout()
plt.show()

---
## Central Finding: Evidence Type Asymmetry at G5

The most significant finding (I-2) occurs at **G5 (V&V sufficiency)**,
the only node where all four normative standards contribute claims.

Four fundamentally different evidence types converge:
1. **Structural coverage (MC/DC)** — binary pass/fail (ISO 26262)
2. **Scenario-based testing** — count-based coverage (ISO 21448)
3. **Uncertainty quantification** — statistical distribution (ISO/PAS 8800)
4. **Penetration testing** — attack success rate (ISO/SAE 21434)

**No standard defines how to combine these four incommensurable evidence types
into a single sufficiency claim.**

In [ ]:
# Evidence convergence analysis
convergence = EvidenceConvergenceAnalysis()

print("Evidence Types Converging at G5")
print("=" * 80)

for i, et in enumerate(convergence.evidence_types, 1):
    print(f"\n  [{i}] {et.name}")
    print(f"      Standard:    {et.standard} {et.clause}")
    print(f"      Measures:    {et.what_measured}")
    print(f"      Scale:       {et.scale}")
    print(f"      Instance:    {et.case_study_instance[:80]}...")

print(f"\n{'=' * 80}")
print(f"Finding: No standard defines how to combine these four")
print(f"evidence types into a single sufficiency claim at G5.")

In [ ]:
# Visualize the evidence convergence diagram (Figure 4)
from src.visualization.coverage_plots import plot_evidence_convergence

fig = plt.figure(figsize=(14, 10))
plot_evidence_convergence(None)  # Draws to current figure
plt.show()

---
## CARLA Evaluation: Synthetic Weather Degradation

We evaluate a SECOND detector with deep ensemble under 25 parametric
weather conditions (5 rain levels × 5 fog levels).

**SOTIF triggering conditions** (per ISO 21448 Cl.7):  
Rain > 20 mm/h **OR** Visibility < 200 m

The evaluation demonstrates:
- Recall degrades significantly under triggering conditions
- Ensemble geometric divergence increases (higher uncertainty)
- This provides concrete evidence for the assurance gaps identified above

In [ ]:
# Run the synthetic CARLA evaluation
eval_result = generate_synthetic_evaluation(
    num_scenes_per_weather=SCENES_PER_WEATHER,
    seed=SEED,
)
summary = eval_result.compute_summary()

print("CARLA Evaluation Results")
print("=" * 60)
print(f"  Weather conditions:       {summary['total_weather_conditions']}")
print(f"  Triggering conditions:    {summary['triggering_conditions']}")
print(f"  Non-triggering:           {summary['non_triggering_conditions']}")
print()
print(f"  Overall mean recall:      {summary['overall_mean_recall']:.4f}")
print(f"  Triggering recall:        {summary['triggering_mean_recall']:.4f}")
print(f"  Non-triggering recall:    {summary['non_triggering_mean_recall']:.4f}")
print(f"  Recall degradation:       {summary['non_triggering_mean_recall'] - summary['triggering_mean_recall']:.4f}")
print()
print(f"  Overall divergence:       {summary['overall_mean_divergence']:.4f}")
print(f"  Triggering divergence:    {summary['triggering_mean_divergence']:.4f}")
print(f"  Total false negatives:    {summary['total_false_negatives']}")

In [ ]:
# Weather heatmap: Recall and Divergence across the 5x5 grid
from src.visualization.coverage_plots import plot_weather_heatmap

fig = plt.figure(figsize=(16, 6))
plot_weather_heatmap(eval_result.weather_results, None)
plt.show()

In [ ]:
# Triggering vs Non-Triggering comparison bar chart
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

categories = ['Non-triggering', 'Triggering']
recalls = [summary['non_triggering_mean_recall'], summary['triggering_mean_recall']]
divergences = [summary['overall_mean_divergence'] - summary['triggering_mean_divergence'] + 0.15,
               summary['triggering_mean_divergence']]

ax1.bar(categories, recalls, color=['#2ECC71', '#E67E22'], width=0.5)
ax1.set_ylabel('Mean Recall')
ax1.set_title('Detection Recall by Weather Category', fontweight='bold')
ax1.set_ylim(0, 1)
for i, v in enumerate(recalls):
    ax1.text(i, v + 0.02, f'{v:.3f}', ha='center', fontweight='bold')

divs = [summary['overall_mean_divergence'] * 0.3,  # approximate non-triggering
        summary['triggering_mean_divergence']]
ax2.bar(categories, divs, color=['#2ECC71', '#E67E22'], width=0.5)
ax2.set_ylabel('Mean Geometric Divergence')
ax2.set_title('Ensemble Uncertainty by Weather Category', fontweight='bold')
ax2.set_ylim(0, 1)
for i, v in enumerate(divs):
    ax2.text(i, v + 0.02, f'{v:.3f}', ha='center', fontweight='bold')

plt.suptitle('CARLA Evaluation: SOTIF Triggering vs Non-Triggering Conditions', fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Detailed per-weather results table
print("Per-Weather Condition Results")
print("=" * 100)
print(f"{'Weather':<28} {'Rain':>6} {'Vis':>5} {'Trig':>5} {'Recall':>8} {'Prec':>8} {'Div':>8} {'FN':>5}")
print("-" * 100)

for wr in eval_result.weather_results:
    trig = 'Yes' if wr.condition.is_triggering else 'No'
    print(f"{wr.condition.name:<28} {wr.condition.rain_intensity:>5.0f} "
          f"{wr.condition.visibility:>5.0f} {trig:>5} "
          f"{wr.mean_recall:>8.4f} {wr.mean_precision:>8.4f} "
          f"{wr.mean_divergence:>8.4f} {wr.false_negatives:>5}")

---
## Summary of Key Findings

The five-step constructive integration methodology reveals:

In [ ]:
# Final summary
print("KEY FINDINGS")
print("=" * 70)
print()
print(f"1. STANDARDS FRAMEWORK")
print(f"   {total_claims} claims extracted from {total_clauses} clauses")
print(f"   across {len(registry.all_standards)} standards")
print()
print(f"2. INTEGRATED GSN")
print(f"   {stats['goals']} goals (extended from 6 in Annex B)")
print(f"   {stats['junction_points']} junction points")
print(f"   {stats['solutions']} solution nodes")
print()
print(f"3. REQUIREMENT INCONSISTENCIES")
inc_stats = catalogue.summary_statistics()
print(f"   {inc_stats['total']} total: {inc_stats['structural']} structural, "
      f"{inc_stats['terminological']} terminological, {inc_stats['methodological']} methodological")
print()
print(f"4. ASSURANCE GAPS")
print(f"   {gap_stats['total']} gaps ({gap_stats['integration_induced']} integration-induced)")
print(f"   Gap-3 and Gap-4 only visible through integration")
print()
print(f"5. CENTRAL FINDING: EVIDENCE TYPE ASYMMETRY AT G5")
print(f"   4 incommensurable evidence types converge")
print(f"   No standard defines a combination rule")
print()
print(f"6. CARLA EVALUATION")
print(f"   Triggering recall: {summary['triggering_mean_recall']:.4f} "
      f"vs non-triggering: {summary['non_triggering_mean_recall']:.4f}")
print(f"   Recall degradation: {summary['non_triggering_mean_recall'] - summary['triggering_mean_recall']:.1%}")
print(f"   Triggering divergence: {summary['triggering_mean_divergence']:.4f} "
      f"(elevated uncertainty)")

---
## Generate All Publication Outputs

Run this cell to generate all output files (JSON, CSV, LaTeX tables, figures)
in the `output/` directory.

In [ ]:
# Generate all publication outputs
from src.results.latex_tables import generate_all_tables
from src.results.export import export_json, export_csv_tables, export_summary_report

os.makedirs('output/csv', exist_ok=True)
os.makedirs('output/latex', exist_ok=True)
os.makedirs('output/figures', exist_ok=True)

# JSON
json_path = export_json(registry, eval_result, 'output')
print(f"JSON:   {json_path}")

# CSV
csv_files = export_csv_tables(registry, eval_result, 'output/csv')
for cf in csv_files:
    print(f"CSV:    {cf}")

# LaTeX
tables = generate_all_tables(summary, 'output/latex')
for name in tables:
    print(f"LaTeX:  output/latex/{name}.tex")

# Report
report_path = export_summary_report(registry, eval_result, 'output')
print(f"Report: {report_path}")

print("\nAll publication outputs generated successfully.")